# Day 2 — DuckDB Medallion Pipeline (6H Grain)

**Goal:** Ingest raw hourly weather and flood data into DuckDB Bronze layer, and process them into 6-hourly Silver and Gold layers.

**Steps:**
1. **Bronze**: Raw API ingestion to `bronze` schema.
2. **Silver**: 6-hourly resampling (SUM/AVG/MAX).
3. **Gold**: Feature engineering (Lags) and Labeling (`is_flood`).

In [1]:
import duckdb
import pandas as pd
from src import ingestion, pipeline, config

## 1. Bronze Ingestion
Fetching full history from APIs and loading into DuckDB raw tables.

In [2]:
# Run full ingestion (API -> DuckDB Bronze)
ingestion.run_full_ingestion()

2026-04-23 10:02:41,532 - INFO - Fetching hourly weather for Coastal (2020-01-01 to 2026-04-20)
2026-04-23 10:02:44,386 - INFO - Successfully ingested 55248 rows into bronze.weather_raw_coastal and saved to /home/aliagabalayev/Desktop/Workspace/Weather-Prediction/data/raw/weather_raw_coastal.parquet
2026-04-23 10:02:44,422 - INFO - Fetching flood data for Coastal (2020-01-01 to 2026-04-20)
2026-04-23 10:02:45,135 - INFO - Successfully ingested 2302 rows into bronze.flood_raw_coastal and saved to /home/aliagabalayev/Desktop/Workspace/Weather-Prediction/data/raw/flood_raw_coastal.parquet
2026-04-23 10:02:45,145 - INFO - Fetching hourly weather for Urban (2020-01-01 to 2026-04-20)
2026-04-23 10:02:54,368 - INFO - Successfully ingested 55248 rows into bronze.weather_raw_urban and saved to /home/aliagabalayev/Desktop/Workspace/Weather-Prediction/data/raw/weather_raw_urban.parquet
2026-04-23 10:02:54,402 - INFO - Fetching flood data for Urban (2020-01-01 to 2026-04-20)
2026-04-23 10:02:56,52

In [3]:
# Verify Bronze tables
conn = duckdb.connect(str(config.DB_PATH))
conn.execute("SHOW ALL TABLES").df()

,database,schema,name,column_names,column_types,temporary
0,weather,bronze,flood_raw_coastal,"[time, river_discharge, zone, _ingested_at, _s...","[TIMESTAMP, DOUBLE, VARCHAR, TIMESTAMP, VARCHAR]",False
1,weather,bronze,flood_raw_highland,"[time, river_discharge, zone, _ingested_at, _s...","[TIMESTAMP, DOUBLE, VARCHAR, TIMESTAMP, VARCHAR]",False
2,weather,bronze,flood_raw_urban,"[time, river_discharge, zone, _ingested_at, _s...","[TIMESTAMP, DOUBLE, VARCHAR, TIMESTAMP, VARCHAR]",False
3,weather,bronze,weather_raw_coastal,"[time, temperature_2m, relative_humidity_2m, p...","[TIMESTAMP, DOUBLE, BIGINT, DOUBLE, DOUBLE, DO...",False
4,weather,bronze,weather_raw_highland,"[time, temperature_2m, relative_humidity_2m, p...","[TIMESTAMP, DOUBLE, BIGINT, DOUBLE, DOUBLE, DO...",False
5,weather,bronze,weather_raw_urban,"[time, temperature_2m, relative_humidity_2m, p...","[TIMESTAMP, DOUBLE, BIGINT, DOUBLE, DOUBLE, DO...",False
6,weather,gold,model_ready_6h,"[zone, time_6h, temperature_2m, relative_humid...","[VARCHAR, TIMESTAMP, DOUBLE, DOUBLE, DOUBLE, D...",False
7,weather,silver,weather_flood_6h,"[zone, time_6h, temperature_2m, relative_humid...","[VARCHAR, TIMESTAMP, DOUBLE, DOUBLE, DOUBLE, D...",False


## 2. Silver & Gold Pipeline
Transforming hourly raw data into 6-hourly model-ready features.

In [4]:
# Run Medallion Pipeline (Bronze -> Silver -> Gold)
pipeline.run_pipeline()

2026-04-23 10:03:18,761 - INFO - Resampling Bronze data to 6-hourly Silver layer...
2026-04-23 10:03:18,816 - INFO - Silver layer saved to /home/aliagabalayev/Desktop/Workspace/Weather-Prediction/data/processed/silver_weather_flood_6h.parquet with 27624 records.


## 3. Data Validation
Checking the integrated model-ready dataset.

In [5]:
df_gold = conn.execute(f"SELECT * FROM {config.SCHEMA_GOLD}.model_ready_6h LIMIT 10").df()

In [6]:
df_floods_only = conn.execute(f"SELECT * FROM {config.SCHEMA_GOLD}.model_ready_6h WHERE is_flood = 1").df()
print(f"Toplam bulunan sel kaydı: {len(df_floods_only)}")
df_floods_only.head()

Toplam bulunan sel kaydı: 524


,zone,time_6h,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,soil_temperature_0_to_7cm,et0_fao_evapotranspiration,river_discharge,daily_max_precip,daily_max_hum,daily_max_discharge,precip_lag_6h,precip_lag_24h,discharge_lag_6h,is_flood
0,Urban,2021-09-27 00:00:00,16.966667,79.166667,0.0,15.450000,0.301667,0.160000,17.616667,0.028333,1.82,0.0,81.0,1.82,0.0,7.2,0.02,1
1,Urban,2021-09-27 06:00:00,18.350000,68.666667,0.0,12.666667,0.295500,0.165833,19.266667,0.160000,1.82,0.0,81.0,1.82,0.0,7.7,1.82,1
2,Urban,2021-09-27 12:00:00,22.783333,52.333333,0.0,11.133333,0.281167,0.166333,25.716667,0.370000,1.82,0.0,81.0,1.82,0.0,0.0,1.82,1
3,Urban,2021-09-27 18:00:00,18.416667,81.000000,0.0,6.450000,0.272167,0.165667,21.866667,0.015000,1.82,0.0,81.0,1.82,0.0,0.0,1.82,1
4,Urban,2021-09-28 00:00:00,17.616667,87.500000,0.1,8.033333,0.268667,0.166833,18.966667,0.001667,2.01,2.1,87.5,2.01,0.0,0.0,1.82,1


In [7]:
# Check Flood Labels Distribution
conn.execute(f"SELECT is_flood, COUNT(*) FROM {config.SCHEMA_GOLD}.model_ready_6h GROUP BY 1").df()

,is_flood,count_star()
0,0,27100
1,1,524


In [8]:
conn.close()